# タイヤ分析：温度分布＆サーモグラフィー

このノートブックでは、タイヤ全体の赤外線センサーからのタイヤ温度データを可視化し、タイヤの挙動を理解してセットアップの問題を特定します。

## このノートブックの内容

- **タイヤ温度ヒートマップ**: ラップ全体にわたる4輪（FL、FR、RL、RR）すべての温度分布を視覚的に表示
- **温度対距離**: コースの各セクションでタイヤ温度がどのように変化するかを確認
- **速度＆Gフォースオーバーレイ**: タイヤ温度と速度および横/縦方向加速度の相関を表示
- **ドライバー入力オーバーレイ**: 温度データに合わせたスロットル、ブレーキ、ステアリング入力を表示

## 結果の解釈方法

### 温度ヒートマップ
- **センサーゾーン**: タイヤ幅全体の温度センサー位置を表す
  - FL/RL: ゾーン1 = 外側（左）、最終ゾーン = 内側（右）
  - FR/RR: ゾーン1 = 内側（左）、最終ゾーン = 外側（右）
- **ホットスポット（明るい）**: 高温エリア - 過大な荷重またはスリップを示す可能性
- **コールドスポット（暗い）**: 使われていないエリア - 使い切れていないグリップの可能性
- **均等な温度勾配**: 良好なタイヤ使用とキャンバー設定を示す

### セットアップの洞察
- **外側エッジが高温**: より大きなネガティブキャンバーが必要かも
- **内側エッジが高温**: ネガティブキャンバーが大きすぎるかも
- **中央が高温**: 空気圧が高すぎる可能性
- **両端が高温、中央が低温**: 空気圧が低すぎる可能性
- **前後の温度差**: アンダーステア/オーバーステア傾向のバランスに関する洞察

## 自分のデータを使用する場合

自分のデータを分析するには：

1. 下の**最初のセルを実行**してパッケージをインストールし、アップロードウィジェットを表示
2. **「Choose File」をクリック**して`.xrk`、`.xrz`、または`.ibt`ファイルを選択
3. **残りのセルをすべて実行**してデータを分析

ステータスインジケーターに使用中のファイルが表示されます。ファイルをアップロードしない場合は、サンプルデータが使用されます。

## 必要なチャンネル

- タイヤ温度チャンネル（タイヤチャンネルピッカーで設定 — AIMの場合: `FL_Ch1`-`FL_Ch8`、iRacingの場合: `LFtempL`/`LFtempM`/`LFtempR`）
- 距離計算用のGPS Speedチャンネル
- 加速度データ（チャンネルピッカーで設定）
- ドライバー入力チャンネル（チャンネルピッカーで設定）

**注意:** このノートブックはJupyterLite（ブラウザ）と通常のJupyterLab環境の両方で動作します。

In [ ]:
# 必要なパッケージをインストール（JupyterLiteで必要、通常のJupyterLabでは既にインストール済みならスキップ）
%pip install -q pandas plotly libxrk libibt motorsports-data-notebook jinja2 ipywidgets

# Rustパーサーバックエンドを使用（ファイル読み込みが約3倍高速）
import os

os.environ["LIBXRK_BACKEND"] = "rust"

# ヘルパー関数をインポート
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_tire_thermography,
    show_fig,
)
from motorsports_data_notebook.widgets import SessionPicker

# セッションピッカーとチャンネル・タイヤ温度設定
# 自分のファイルをアップロードして分析するラップを選択
# タイヤ温度チャンネルは車両プロファイルから自動設定されます
session = SessionPicker(
    default_file="../data/CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz",
    channel_mapping={
        "lateral_g": "LateralAcc",
        "inline_g": "InlineAcc",
        "throttle": "PPS",
        "brake": "BrakePress",
        "steering": "SteerAngle",
    },
    tire_channel_mapping={
        "FL": [f"FL_Ch{i}" for i in range(1, 9)],
        "FR": [f"FR_Ch{i}" for i in range(1, 9)],
        "RL": [f"RL_Ch{i}" for i in range(1, 9)],
        "RR": [f"RR_Ch{i}" for i in range(1, 9)],
    },
)
session.display()

In [ ]:
# ラップ情報をpandas DataFrameとして取得
laps = session.get_laps()

In [ ]:
# ラップタイム一覧を表示
laps.style.format({"lap_time": format_lap_time})  # type: ignore[dict-item]

In [ ]:
# 設定されたチャンネル名とセッションデータを取得
log = session.get_log()
CHANNEL_NAMES = session.get_channel_names()

# タイヤ温度チャンネルを収集 — ログに存在するチャンネルのみ
available = set(log.channels.keys())
tire_channels = [
    v for k, v in CHANNEL_NAMES.items() if k.startswith("tire_temp_") and v in available
]

other_channels = [
    "distance_m",
    "speed_kmh",
    CHANNEL_NAMES["lateral_g"],
    CHANNEL_NAMES["inline_g"],
    CHANNEL_NAMES["brake"],
    CHANNEL_NAMES["throttle"],
    CHANNEL_NAMES["steering"],
]

# libxrk 0.5.0のメソッドを使用して選択したラップのデータを抽出
selected_lap = session.get_selected_lap()
lap_num = int(selected_lap["num"])

# ラップでフィルタし、チャンネルを選択し、distance_m時間軸にリサンプル
channels = (
    log.filter_by_lap(lap_num)
    .select_channels(tire_channels + other_channels)
    .resample_to_channel("distance_m")
    .channels
)

In [ ]:
# タイヤサーモグラフィー - 選択したラップ
fig = plot_tire_thermography(
    channels, CHANNEL_NAMES, title=f"タイヤ温度 - ラップ {int(selected_lap['num'])}"
)
show_fig(fig)